<a href="https://colab.research.google.com/github/shreeshbhat04-ctrl/Kaggle-repo/blob/main/Copy_of_Llama_3_1_8B_Instruct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers bitsandbytes>=0.46.1

## Local Inference on GPU
Model page: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# Clear CUDA cache

# Configure 8-bit quantization with CPU offloading allowed
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model_id = "meta-llama/Llama-3.1-8B-Instruct"

# Load the model and tokenizer
# We let device_map="auto" handle the split, but with the offload flag enabled
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Create the pipeline
pip = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

messages = [
    {"role": "user", "content": "On the scale of 1 to 10 ,how much do u know and can write latex?"},
]

# Generate output
print(pip(messages, max_new_tokens=128))

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[{'generated_text': [{'role': 'user', 'content': 'On the scale of 1 to 10 ,how much do u know and can write latex?'}, {'role': 'assistant', 'content': "I'd rate my ability to write LaTeX as an 8 out of 10. I have been trained on a wide range of mathematical and scientific topics, and I can generate LaTeX code for various mathematical expressions, equations, and formulas.\n\nI can write LaTeX code for:\n\n1. Basic mathematical expressions (e.g., $\\frac{x^2 + 3x - 4}{x + 1}$)\n2. Equations (e.g., $E = mc^2$)\n3. Matrices and tensors (e.g., $\\begin{bmatrix} a & b \\\\ c & d \\end{bmatrix}$)\n"}]}]


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


KeyboardInterrupt: 

## Remote Inference via Inference Providers
Ensure you have a valid **HF_TOKEN** set in your environment. You can get your token from [your settings page](https://huggingface.co/settings/tokens). Note: running this may incur charges above the free tier.
The following Python example shows how to run the model remotely on HF Inference Providers, automatically selecting an available inference provider for you.
For more information on how to use the Inference Providers, please refer to our [documentation and guides](https://huggingface.co/docs/inference-providers/en/index).

In [ ]:
import os
os.environ['HF_TOKEN'] = 'YOUR_TOKEN_HERE'

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

completion = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message)

here are the templates :\documentclass[conference]{IEEEtran}
\usepackage{cite}
\usepackage{amsmath,amssymb,amsfonts}
\usepackage{algorithmic}
\usepackage{graphicx}
\usepackage{textcomp}
\usepackage{xcolor}

\begin{document}

\title{[TITLE]}

\author{\IEEEauthorblockN{[AUTHOR NAME]}
\IEEEauthorblockA{\textit{[DEPARTMENT]} \\
\textit{[ORGANIZATION]}\\
[CITY, COUNTRY] \\
[EMAIL]}
}

\maketitle

\begin{abstract}
[ABSTRACT CONTENT]
\end{abstract}

\begin{IEEEkeywords}
[KEYWORD 1], [KEYWORD 2], [KEYWORD 3]
\end{IEEEkeywords}

\section{Introduction}
[INTRODUCTION CONTENT]

\section{[SECTION TITLE]}
[SECTION CONTENT]

\subsection{[SUBSECTION TITLE]}
[SUBSECTION CONTENT]

\begin{figure}[htbp]
\centerline{\includegraphics{[IMAGE_NAME.png]}}
\caption{[FIGURE CAPTION]}
\label{[FIGURE_LABEL]}
\end{figure}

\begin{table}[htbp]
\caption{[TABLE CAPTION]}
\begin{center}
\begin{tabular}{|c|c|c|}
\hline
[COL 1] & [COL 2] & [COL 3] \\
\hline
[DATA] & [DATA] & [DATA] \\
\hline
\end{tabular}
\label{[TABLE_LABEL]}
\end{center}
\end{table}

\section*{Acknowledgment}
[ACKNOWLEDGMENT CONTENT]

\begin{thebibliography}{00}
\bibitem{b1} [CITATION CONTENT]
\end{thebibliography}

\end{document}     ----------------->second one:
\documentclass[conference]{IEEEtran}
\usepackage{cite}
\usepackage{amsmath,amssymb,amsfonts}
\usepackage{algorithmic}
\usepackage{graphicx}
\usepackage{textcomp}
\usepackage{xcolor}
\usepackage{listings}
\usepackage{hyperref}

\lstset{
  basicstyle=\ttfamily\small,
  breaklines=true,
  frame=single,
  language=C++,
  commentstyle=\color{gray},
  keywordstyle=\color{blue},
}

\begin{document}

\title{Design and Implementation of Bluetooth Low Energy (BLE) Communication using ESP32}

\author{\IEEEauthorblockN{Shreesha H B (1DS23EC200), Soundarya D Shetty (1DS23EC214), Sreeshanth K Prakash (1DS23EC215)}
\IEEEauthorblockA{\textit{Department of Electronics \& Communication Engineering} \\
\textit{Dayananda Sagar College of Engineering}\\
(An Autonomous Institute affiliated to Visvesvaraya Technological University (VTU), Belagavi, \\
Approved by AICTE and UGC, Accredited by NAAC with `A' grade \& ISO 9001 -- 2015 Certified Institution) \\
Shavige Malleshwara Hills, Kumaraswamy Layout, Bengaluru-560 078, India \\
Under the Guidance of Dr. Naveen Gowda}
}

\maketitle

\begin{abstract}
Bluetooth Low Energy (BLE) is a widely used wireless communication technology designed specifically for low-power Internet of Things (IoT) applications. It enables efficient data transmission over short distances while consuming minimal energy, making it highly suitable for battery-operated devices. In this experiment, the ESP32 microcontroller is configured to operate as a BLE server that broadcasts a custom service and characteristic. A smartphone or BLE-enabled device acts as the client, which scans, connects, and interacts with the ESP32.

The experiment focuses on establishing a stable BLE connection, enabling reliable data exchange through read and write operations, and understanding the working principles of BLE protocols such as GATT (Generic Attribute Profile) and GAP (Generic Access Profile). The implementation also demonstrates how services and characteristics are used to structure and transmit data in a BLE-based system.

The developed system demonstrates efficient short-range wireless communication with minimal power consumption and stable connectivity. The results confirm that ESP32 can reliably transmit and receive data using BLE, making it suitable for a wide range of real-time applications such as smart home automation, wearable devices, healthcare monitoring, and industrial IoT systems. This experiment provides a practical understanding of BLE communication and its role in modern IoT-based solutions.
\end{abstract}

\begin{IEEEkeywords}
Bluetooth Low Energy (BLE), ESP32, IoT, Wireless Communication, GATT Protocol
\end{IEEEkeywords}

\tableofcontents
\newpage

\section*{Certificate}
This is to certify that the Open ended experiment entitled \textbf{``Demonstrate how to use BLE with ESP32''} as part of \textbf{Internet of Things lab (22ECL671)} is a bonafide work carried out by \textbf{Shreesha H B (1DS23EC200)}, \textbf{Soundarya D Shetty (1DS23EC214)} \& \textbf{Sreeshanth K Prakash (1DS23EC215)}, as 30-mark component in partial fulfillment for the 6\textsuperscript{th} semester of Bachelor of Engineering in Electronics and Communication Engineering of the Visvesvaraya Technological University, Belagavi during the year 2025-2026. The Open Ended Experiment report has been approved as it satisfies the academic requirements prescribed for the Bachelor of Engineering degree.

\vspace{2em}

\noindent\textbf{Signature of Faculty} \hfill \textbf{Signature of HOD} \\
\textbf{Dr. Naveen Gowda} \hfill \textbf{Dr. Shobha K R}

\newpage

\section*{Declaration}

We declare that we abide by the ethical principles and commit to professional ethics and responsibilities and norms of the engineering practice. The work submitted in this report of \textbf{Internet of Things lab (22ECL671)}, 6\textsuperscript{th} Semester BE, ECE has been compiled by referring to the relevant online and offline resources to the best of our understanding and in partial fulfillment of the requirement for the award of the degree of Bachelor of Engineering in Electronics and Communication Engineering, at Dayananda Sagar College of Engineering, an autonomous institution affiliated to VTU, Belagavi during the academic year 2025-2026.

We hereby declare that the same has not been submitted in part or full for other academic purposes.

\vspace{2em}

\noindent 1DS23EC200 \quad SHREESHA H B \\
1DS23EC214 \quad SOUNDARYA D SHETTY \\
1DS23EC215 \quad SREESHANTH K PRAKASH

\vspace{1em}

\noindent\textbf{Place:} Bangalore \\
\textbf{Date:} 21-04-2026

\newpage

\section*{Acknowledgment}

It is a great pleasure for us to acknowledge the assistance and support of many individuals who have been responsible for the successful completion of this project.

We take this opportunity to express our sincere gratitude to Dayananda Sagar College of Engineering for giving us the opportunity to pursue our Bachelor's Degrees in this institution. In particular, we would like to thank Dr. B G Prasad, Principal of Dayananda Sagar College of Engineering, for his constant encouragement and advice.

We would like to express our gratitude to Dr. Shobha K R, Professor and HoD, Department of Electronics and Communication Engineering, Dayananda Sagar College of Engineering, for her motivation and invaluable support throughout the development of this project.

We extend our heartfelt thanks to Dr. Naveen Kumar, for her constant guidance, encouragement, and unwavering support throughout this project. Her patience in clarifying our doubts and her valuable insights during the lab sessions played a crucial role in shaping our understanding and successfully completing the work. We are truly grateful for the time and effort she dedicated to ensuring our progress and learning.

\vspace{2em}

\hfill 1DS23EC200 SHREESHA H B \\
\hfill 1DS23EC214 SOUNDARYA D SHETTY \\
\hfill 1DS23EC215 SREESHANTH K PRAKASH

\newpage

\section{Introduction}

The rapid growth of the Internet of Things (IoT) has increased the demand for low-power, cost-effective, and efficient wireless communication technologies. Bluetooth Low Energy (BLE) has emerged as a key solution due to its ability to provide reliable communication while consuming significantly less power compared to traditional Bluetooth. BLE operates in the 2.4 GHz ISM band and is specifically optimized for transmitting small amounts of data over short distances.

The ESP32 microcontroller, developed by Espressif Systems, is a powerful and versatile device that supports both Wi-Fi and BLE connectivity. Its built-in BLE functionality makes it an ideal choice for developing IoT-based wireless communication systems without the need for additional hardware modules.

BLE communication is based on a client-server architecture. The server (ESP32) hosts data in the form of services and characteristics, while the client (smartphone or other BLE device) accesses this data. Two important protocols used in BLE are:

\begin{itemize}
    \item \textbf{Generic Access Profile (GAP):} Responsible for device discovery, connection establishment, and advertising.
    \item \textbf{Generic Attribute Profile (GATT):} Defines how data is structured and exchanged between devices.
\end{itemize}

In this experiment, the ESP32 is programmed as a BLE server that advertises its presence to nearby devices. A mobile application is used to scan and connect to the ESP32, allowing the user to read and write data through BLE characteristics. This setup helps in understanding the practical implementation of BLE communication, including device pairing, service creation, and data transmission.

\section{Connection Diagram}

The ESP32 is connected to the computer via USB for programming and serial monitoring. No additional external components are required since the ESP32 has a built-in BLE module.

% \begin{figure}[htbp]
% \centerline{\includegraphics[width=0.8\columnwidth]{connection_diagram.png}}
% \caption{Connection Diagram for ESP32 BLE Setup}
% \label{fig:connection}
% \end{figure}

\section{Program}

The following Arduino code configures the ESP32 as a BLE server with a custom service and characteristic:

\begin{lstlisting}[caption={ESP32 BLE Server Code}]
#include <BLEDevice.h>    // Library to handle BLE device functions
#include <BLEUtils.h>     // Provides utility functions for BLE operations
#include <BLEServer.h>    // Used to create and manage BLE server

// Unique IDs for BLE service and characteristic
#define SERVICE_UUID        "4fafc201-1fb5-459e-8fcc-c5c9c331914b"
#define CHARACTERISTIC_UUID "beb5483e-36e1-4688-b7f5-ea07361b26a8"

void setup() {
  Serial.begin(115200);                   // Starts serial communication for debugging
  Serial.println("Starting BLE...");      // Prints message to show BLE initialization

  BLEDevice::init("MyESP32");            // Initialize ESP32 as BLE device with name

  BLEServer *pServer = BLEDevice::createServer();  // Creates BLE server instance

  BLEService *pService = pServer->createService(SERVICE_UUID);  // Creates a BLE service using UUID

  BLECharacteristic *pCharacteristic = pService->createCharacteristic(
    CHARACTERISTIC_UUID,                   // Assigns UUID to characteristic
    BLECharacteristic::PROPERTY_READ |     // Enables read property
    BLECharacteristic::PROPERTY_WRITE      // Enables write property
  );                                       // Ends characteristic creation

  pCharacteristic->setValue("Hello World!");  // Sets initial data value

  pService->start();                       // Starts the BLE service

  BLEAdvertising *pAdvertising = BLEDevice::getAdvertising();  // Gets advertising object
  pAdvertising->addServiceUUID(SERVICE_UUID);  // Adds service UUID to advertisement
  pAdvertising->setScanResponse(true);         // Enables response when scanned
  BLEDevice::startAdvertising();               // Starts BLE advertising

  Serial.println("BLE device is now visible");  // Confirms device is ready to connect
}

void loop() {          // Loop function runs continuously
  delay(2000);         // Small delay (2 sec), no continuous task needed
}
\end{lstlisting}

\section{Applications}

BLE communication using ESP32 has a wide range of practical applications:

\subsection{Smart Home Automation}
\begin{itemize}
    \item BLE can control lights, fans, and appliances using a mobile phone.
    \item Example: Turning ON/OFF LED using BLE app.
\end{itemize}

\subsection{Health Monitoring Systems}
\begin{itemize}
    \item BLE is used in wearable devices like fitness bands.
    \item Data such as heart rate or temperature is sent to phone.
    \item ESP32 can be connected to sensors and transmit health data.
\end{itemize}

\subsection{Industrial IoT Monitoring}
\begin{itemize}
    \item Machines can send status updates wirelessly.
    \item Maintenance alerts can be monitored remotely.
    \item Reduces wiring complexity.
\end{itemize}

BLE concepts such as services, characteristics, GATT, and GAP demonstrate practical use in IoT systems. Hence, BLE can be effectively applied in real-time applications such as smart home automation, healthcare monitoring, and wireless sensor networks.

\section{Results and Conclusion}

The ESP32 BLE module was successfully implemented and tested in this experiment. The device was able to advertise itself and was detected by a smartphone using a BLE scanning application.

A stable connection was established between the ESP32 (server) and the mobile device (client), enabling successful data exchange through read and write operations. The characteristic value was correctly transmitted and updated without any errors during multiple test cases, including device discovery, connection, and data transfer.

The results confirm that BLE communication using ESP32 is reliable, energy-efficient, and suitable for short-range wireless applications. This experiment provides a clear understanding of BLE concepts such as services, characteristics, GATT, and GAP, and demonstrates its practical use in IoT systems.

The experiment provides a strong foundation for developing advanced IoT applications such as smart control systems, wearable technology, and wireless sensor networks.

\begin{thebibliography}{00}
\bibitem{b1} ESP32 BLE Tutorial -- Last Minute Engineers \\
\url{https://lastminuteengineers.com/esp32-ble-tutorial/}

\bibitem{b2} ESP32 BLE Overview -- Electronics Hub \\
\url{https://www.electronicshub.org/esp32-ble/}

\bibitem{b3} Espressif Systems -- Official Documentation \\
\url{https://docs.espressif.com/projects/esp-idf/en/latest/esp32/api-reference/bluetooth/index.html}

\bibitem{b4} Arduino ESP32 BLE Library Documentation \\
\url{https://www.arduino.cc/reference/en/libraries/esp32-ble/}

\bibitem{b5} Bluetooth Special Interest Group (Bluetooth SIG) \\
\url{https://www.bluetooth.com/learn-about-bluetooth/}

\bibitem{b6} Random Nerd Tutorials -- ESP32 BLE Examples \\
\url{https://randomnerdtutorials.com/esp32-ble-server-client/}
\end{thebibliography}

\end{document}


In [ ]:

!pip install groq

import json
import time
import os
from groq import Groq
from google.colab import userdata

# 1. Setup API Client
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=GROQ_API_KEY)
except Exception as e:
    print(f"Error: Make sure GROQ_API_KEY is set in Colab Secrets. {e}")

# 2. The LaTeX Template (from your notebook context)
LATEX_TEMPLATE = r"""
\documentclass[conference]{IEEEtran}
\usepackage{cite}
\usepackage{amsmath,amssymb,amsfonts}
\usepackage{algorithmic}
\usepackage{graphicx}
\usepackage{textcomp}
\usepackage{xcolor}

\begin{document}

\title{[TITLE]}

\author{\IEEEauthorblockN{[AUTHOR NAME]}
\IEEEauthorblockA{\textit{[DEPARTMENT]} \\
\textit{[ORGANIZATION]}\\
[CITY, COUNTRY] \\
[EMAIL]}
}

\maketitle

\begin{abstract}
[ABSTRACT CONTENT]
\end{abstract}

\begin{IEEEkeywords}
[KEYWORD 1], [KEYWORD 2], [KEYWORD 3]
\end{IEEEkeywords}

\section{Introduction}
[INTRODUCTION CONTENT]

\section{[SECTION TITLE]}
[SECTION CONTENT]

\subsection{[SUBSECTION TITLE]}
[SUBSECTION CONTENT]

\begin{figure}[htbp]
\centerline{\includegraphics{[IMAGE_NAME.png]}}
\caption{[FIGURE CAPTION]}
\label{[FIGURE_LABEL]}
\end{figure}

\begin{table}[htbp]
\caption{[TABLE CAPTION]}
\begin{center}
\begin{tabular}{|c|c|c|}
\hline
[COL 1] & [COL 2] & [COL 3] \\
\hline
[DATA] & [DATA] & [DATA] \\
\hline
\end{tabular}
\label{[TABLE_LABEL]}
\end{center}
\end{table}

\section*{Acknowledgment}
[ACKNOWLEDGMENT CONTENT]

\begin{thebibliography}{00}
\bibitem{b1} [CITATION CONTENT]
\end{thebibliography}

\end{document}
"""

# 3. Generation Function
def generate_synthetic_data(iteration):
    system_prompt = (
        "You are an expert LaTeX typesetter . "
        "Your goal is to generate training data for a student model."
    )

    user_prompt = f"""Generate a synthetic data pair for an lab report making.

    Step 1: Create a 'markdown_input'. This should be a messy, fragmented set of student notes about an ESP32 or IoT project. Use bullet points, informal language, and missing details.

    Step 2: Create a 'latex_output'. Map the data from the messy notes into this exact LaTeX template:
    {LATEX_TEMPLATE}

    Ensure the LaTeX is perfectly formatted and valid.

    Return exactly a JSON object with keys: 'markdown_input' and 'latex_output'.
    """

    try:
        completion = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.7,
        )
        return json.loads(completion.choices[0].message.content)
    except Exception as e:
        print(f"Error on iteration {iteration}: {e}")
        return None

# 4. Execution Loop
output_file = "vtu_iot_dataset_llama.jsonl"
num_samples = 50

print(f"Starting generation of {num_samples} samples...")

with open(output_file, "a") as f:
    for i in range(num_samples):
        data = generate_synthetic_data(i + 1)
        if data:
            f.write(json.dumps(data) + "\n")
            print(f"Generated sample {i+1}/{num_samples}")

        # Sleep to avoid rate limits
        time.sleep(2)

print(f"Done! Dataset saved to {output_file}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 7.9 MB/s eta 0:00:00
Starting generation of 50 samples...
Generated sample 1/50
Generated sample 2/50
Generated sample 3/50
Generated sample 4/50
Generated sample 5/50
Generated sample 6/50
Generated sample 7/50
Generated sample 8/50
Generated sample 9/50
Generated sample 10/50
Generated sample 11/50
Generated sample 12/50
Generated sample 13/50
Generated sample 14/50
Generated sample 15/50
Generated sample 16/50
Error on iteration 17: Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': '{\n  "markdown_input": "* ESP32 IoT project:\\n * Using Wi-Fi module to connect to the internet\\n * Using sensor to detect temperature and humidity\\n * Using OLED display to display data\\n * Need to write code for data analysis\\n * Need to add more features to the project\

In [ ]:
# DATA GENERATION FOR TEMPLATE 2: DETAILED BLE/ESP32 REPORT
import json
import time
from groq import Groq
from google.colab import userdata

try:
    client = Groq(api_key=userdata.get('GROQ_API_KEY'))
except Exception as e:
    print("Error: GROQ_API_KEY not found in Secrets.")

DETAILED_TEMPLATE = r"""
\documentclass[conference]{IEEEtran}
\usepackage{cite}
\usepackage{amsmath,amssymb,amsfonts}
\usepackage{listings}
\usepackage{hyperref}

\begin{document}
\title{Design and Implementation of BLE Communication using ESP32}
\author{\IEEEauthorblockN{Shreesha H B, Soundarya D Shetty, Sreeshanth K Prakash}}
\maketitle

\begin{abstract} [ABSTRACT_TEXT] \end{abstract}
\begin{IEEEkeywords} [KEYWORDS] \end{IEEEkeywords}

\section*{Certificate} [CERTIFICATE_DATA]
\section*{Declaration} [DECLARATION_DATA]
\section*{Acknowledgment} [ACKNOWLEDGMENT_DATA]
\section{Introduction} [INTRO_DATA]
\section{Program}
\begin{lstlisting}[caption={ESP32 BLE Code}]
[SOURCE_CODE]
\end{lstlisting}
\section{Applications} [APPS_DATA]
\section{Results and Conclusion} [CONCLUSION_DATA]
\begin{thebibliography}{00} [CITATIONS] \end{thebibliography}
\end{document}
"""

def generate_detailed(i):
    system_prompt = """You are an expert LaTeX Architect.
Generate JSON with 'markdown_input' and 'latex_output'.
In 'latex_output', you MUST populate the template provided.
IMPORTANT: Be concise in the content to avoid JSON parsing errors.
Escape all backslashes correctly for JSON strings."""

    user_prompt = f"""Generate a synthetic training pair.
1. markdown_input: Messy student notes about ESP32 BLE project.
2. latex_output: Populate this template: {DETAILED_TEMPLATE}

Return ONLY a valid JSON object."""

    try:
        completion = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"}
        )
        return json.loads(completion.choices[0].message.content)
    except Exception as e:
        print(f"Sample {i+1} failed API check: {e}")
        return None

with open("dataset_detailed_enhanced.jsonl", "a") as f:
    for i in range(25):
        data = generate_detailed(i)
        if data:
            f.write(json.dumps(data) + "\n")
            print(f"Enhanced Detailed Sample {i+1} done")
        time.sleep(1.5)

Enhanced Detailed Sample 1 done
Enhanced Detailed Sample 2 done
Sample 3 failed API check: Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': '{\n  "markdown_input":\n      "### ESP32 BLE Project\\n#### Introduction\\nESP32 BLE project is used for developing a low-power BLE device.\\n- BLE module used: ESP32.\\n- Programming language: C++.\\n\\n#### Code Explanation\\n- We have used ESP32 BLE code for developing the BLE device.\\n- The code used in this project is provided below:\\n\\n```c\\n#include <WiFi.h>\\n#include <BLEDevice.h>\\n\\n#define SERVICE_UUID        "4fafc201-1fb2-459e-8fcc-c5c582d71fb2"\n#define CHARACTERISTIC_UUID "beb5483e-36c1-4688-b7f5-e2421f287a8c"\n\nclass MyBTDevice{\n  public:\n    void setup() {\n      Serial.begin(115200);\n      BLE.begin();\n    }\n    void loop() {\n      if (BLE.connec